# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uzairrateef-rgb/my-flyrank-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [22]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
y = df["is_declining_label"].values

print(df.shape[0], "pages loaded")

30000 pages loaded


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"My lane is Content Refresh Prioritization, I'm not sorting pages into two fixed buckets (classification) or grouping similar pages together (clustering); I'm ordering all pages by how urgently each one needs review, so a content team can work down the list from the top. That's what Notebook 2's precision_at_k function was built for, evaluating a ranking, not a single prediction. Scoring (a raw urgency number per page) is closely related, but ranking better matches the actual decision: 'give me the top 50 to review this sprint,' not tell me page X's exact score."

"My lane is Content Refresh Prioritization, I'm not sorting pages into two fixed buckets (classification) or grouping similar pages together (clustering); I'm ordering all pages by how urgently each one needs review, so a content team can work down the list from the top. That's what Notebook 2's precision_at_k function was built for, evaluating a ranking, not a single prediction. Scoring (a raw urgency number per page) is closely related, but ranking better matches the actual decision: 'give me the top 50 to review this sprint,' not tell me page X's exact score."

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"Proxy target: is_declining_label — 1 if trend_direction == 'down', else 0. This is a defined rule, not a directly observed business outcome (we don't have 'this page was manually flagged and fixed by a human,' so we substitute a measurable stand-in). It's an honest proxy: pages trending down are the ones a refresh workflow cares about, but the label doesn't capture why they're declining or whether a refresh would actually help."


"Proxy target: is_declining_label — 1 if trend_direction == 'down', else 0. This is a defined rule, not a directly observed business outcome (we don't have 'this page was manually flagged and fixed by a human,' so we substitute a measurable stand-in). It's an honest proxy: pages trending down are the ones a refresh workflow cares about, but the label doesn't capture why they're declining or whether a refresh would actually help."

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"Metric: Precision@50. Of the top 50 pages the model/ranking flags, what fraction are actually declining? This is the right metric because a content team can only review a limited number of pages per cycle precision at a realistic cutoff (50) tells us how much of their limited time would actually go to real problems, rather than false alarms. On this run, the hand rule scores 0.680 at Precision@50, meaning 34 of the top 50 flagged pages are genuinely declining. That's a reasonable baseline, but it also means roughly 1 in 3 flagged pages would waste a reviewer's time."

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

print(f"Precision@50 for the hand rule: {precision_at_k(df['hand_rule_score'], y, 50):.3f}")


Precision@50 for the hand rule: 0.680


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"One row = one piece of published content (content_id) a single page/article belonging to a single client. Each row carries that page's traffic, engagement, and freshness signals over the last 90 days."

cols = ["content_id", "client_id", "content_type", "days_since_last_update",
        "impressions_90d", "avg_position", "ctr", "trend_direction"]
df[cols].head(5)

,content_id,client_id,content_type,days_since_last_update,impressions_90d,avg_position,ctr,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,20,3803,10.6,0.76,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,25,15320,20.3,0.05,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,20,12581,36.5,0.09,down
3,content_331d6c4de07b,client_19581e27de,keyword article,22,11751,6.2,0.49,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,14,19140,44.0,0.13,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
"A fixed rule like 'stale AND visible' only checks two thresholds — it can't weigh how stale against how visible, or notice that the right threshold differs by content_type or avg_position. On this run, the hand rule (0.680) actually outperformed the depth-2 tree (0.600) at Precision@50 which matches the notebook's own warning that a shallow tree gives very few distinct scores, creating large tied blocks where a sharp hand rule can win at the very top of the list. This doesn't mean ML never beats the rule here it means a depth-2 tree is too constrained to prove the point on this metric alone; the earlier notebook showed the tree's advantage tends to show up deeper in the ranking, where the hand rule's simple thresholds run out of signal entirely. Being honest about which one actually won on this run rather than assuming 'ML always wins' is the whole point of this exercise."

# Evidence from Notebook 2: hand rule vs. learned model comparison
from sklearn.tree import DecisionTreeClassifier
features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X, y)
tree_score = tree.predict_proba(X)[:, 1]
print(f"Precision@50 — hand rule: {precision_at_k(df['hand_rule_score'], y, 50):.3f}   "
      f"vs   tree: {precision_at_k(tree_score, y, 50):.3f}")

Precision@50 — hand rule: 0.680   vs   tree: 0.600


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.